# 2. Сопоставление товаров

Сначала отбираем разумные пары-кандидаты, затем оцениваем их сходство. Это сокращает
число сравнений и делает решение понятным для ручной проверки.

Код перенесён из рабочего ноутбука в `scripts/matching.py` с сохранением алгоритмов, порогов и весов.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Работает при запуске Jupyter из корня проекта и из папки notebooks.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'processed').is_dir():
    raise FileNotFoundError('Открой Jupyter из корня проекта.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Предварительный отбор — blocking

Объединяются правила по бренду, типу товара, размеру и информативным словам. Дополнительные
правила TF-IDF возвращают часть пар, потерянных из-за пропусков или различий в написании.
При конфликте заполненных `grade` пара исключается. Отбор кандидата ещё не означает совпадение.

In [2]:
from scripts.matching import run_matching
scored_pairs, blocking_summary = run_matching()
blocking_summary.loc[blocking_summary['scope'].isin(['single_rule', 'union', 'delta_vs_v1'])]

product_type + exact_size -> pairs: 31013
brand + product_type -> pairs: 6766
brand + exact_size -> pairs: 6370
product_type + informative_token -> pairs: 26296
product_type + similar_brand_fragment -> pairs: 8891
similar_brand_fragment + exact_size -> pairs: 7742


Candidate pairs: 53436; saved to data/interim/matching/


,scope,rule_name,pair_count,unique_sklavenitis_items,unique_alphamega_items
0,single_rule,product_type + exact_size,31013.0,1680,3330.0
2,single_rule,brand + product_type,6766.0,1182,1904.0
4,single_rule,brand + exact_size,6370.0,1369,2012.0
6,single_rule,product_type + informative_token,26296.0,1762,3745.0
8,single_rule,product_type + similar_brand_fragment,8891.0,1307,2096.0
10,single_rule,similar_brand_fragment + exact_size,7742.0,1495,2209.0
12,single_rule,tfidf_0_60 + size_similarity_0_70,1515.0,986,1143.0
14,single_rule,tfidf_0_85_any_size,616.0,444,524.0
16,union,all_rules_union_v2,53436.0,2515,5366.0
17,delta_vs_v1,new_pairs_added_by_v2,119.0,31,52.0


## Прозрачная оценка сходства

Используется TF-IDF по символам 3–5 и отдельным словам, `SequenceMatcher` для бренда и типа товара,
а также заданные вручную оценки сходства фасовки. Здесь fuzzy matching — сравнение строк через
стандартный `difflib`, отдельная библиотека fuzzywuzzy не использовалась.

```text
base_score = 0.25 × char_TFIDF + 0.20 × word_TFIDF
           + 0.20 × type_similarity + 0.25 × size_similarity
           + 0.10 × brand_similarity
final_score = clip(base_score + brand_penalty + grade_adjustment, 0, 1)
```

Штраф за явно разные заполненные бренды: −0,15 при сходстве < 0,30.
Поправка за заполненные grade: +0,05 при совпадении и −0,03 при различии.
Скоринг не является вероятностью совпадения.

In [3]:
scored_pairs[['product_name_sk', 'product_name_am', 'size_key_sk', 'size_key_am',
              'final_match_score']].head(12)

,product_name_sk,product_name_am,size_key_sk,size_key_am,final_match_score
0,ZWAN Chicken Luncheon Meat 200g 120903,Zwan Chicken Luncheon Meat 200 g,200_g,200_g,1.0
1,NIVEA Men Sensitive After Shave Balsam 100ml 1...,Nivea Men Sensitive After Shave Balsam 100 ml,100_ml,100_ml,1.0
2,TULIP Chopped Ham 200g 119049,Tulip Chopped Ham 200 g,200_g,200_g,1.0
3,TILDA Basmati Rice 1kg 119976,Tilda Basmati Rice 1 kg,1000_g,1000_g,1.0
4,VEPO Gouda Cheese Slices 200g 224901,Vepo Gouda Cheese Slices 200 g,200_g,200_g,1.0
5,KERRYGOLD Spreadable Butter with Olive Oil 212...,Kerrygold Spreadable Butter With Olive Oil 212 g,212_g,212_g,1.0
6,MELISSA Spaghetti No.6 500g 121939,Melissa Spaghetti No.6 500 g,500_g,500_g,1.0
7,AGROKIPOS HALVATZI Prepacked Greek Cooked Chic...,Agrokipos Halvatzi Prepacked Greek Cooked Chic...,300_g,300_g,1.0
8,KARLA Corned Beef 340g 139954,Karla Corned Beef 340 g,340_g,340_g,1.0
9,LACKMANN Classic Boiled Sausages 450g 784065,Lackmann Classic Boiled Sausages 450 g,450_g,450_g,1.0


In [4]:
pd.DataFrame({'показатель': ['пар-кандидатов', 'товаров Sklavenitis с кандидатами'],
              'значение': [len(scored_pairs), scored_pairs['item_id_sk'].nunique()]})

,показатель,значение
0,пар-кандидатов,53436
1,товаров Sklavenitis с кандидатами,2515


## Где заканчивается автоматизация

Ручные метки переносились на актуальные пары. После ML-экспериментов автор проверяла спорные пары,
дополняла разметку по скорингу и разрешала конфликты «несколько кандидатов на один товар» с помощью
карточек и изображений товаров. Итог — 841 пара один к одному.

Этот ноутбук воспроизводит **кандидатов и скоринг**, а не заменяет ручную проверку новым автоматическим правилом.